# 04 — Évaluation complète du modèle
> Métriques sur le set de test : accuracy, AUC-ROC, F1, matrice de confusion, analyse des erreurs.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from config import MODEL_PATH, CLASS_NAMES
from src.model import load_model
from src.dataset import get_dataloaders
from src.evaluate import (
    get_predictions, plot_confusion_matrix, plot_roc_curve,
    print_classification_report, find_misclassified, full_evaluation
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

model = load_model(MODEL_PATH, device)
_, _, test_loader, class_names = get_dataloaders(num_workers=0)

## 1. Rapport de classification

In [ ]:
labels, preds, probs = get_predictions(model, test_loader, device)
print_classification_report(labels, preds, class_names)

## 2. Matrice de confusion

In [ ]:
plot_confusion_matrix(labels, preds, class_names,
                     save_path='../models/confusion_matrix.png')

## 3. Courbe ROC — AUC
> L'AUC-ROC est **la métrique principale** en classification médicale.
> AUC > 0.90 = excellent | AUC > 0.85 = bon | AUC > 0.75 = acceptable

In [ ]:
roc_auc = plot_roc_curve(labels, probs, save_path='../models/roc_curve.png')
print(f"\nAUC-ROC = {roc_auc:.4f}")
if roc_auc >= 0.90:
    print("✓ Excellent — AUC ≥ 0.90")
elif roc_auc >= 0.85:
    print("✓ Bon — AUC ≥ 0.85")
else:
    print("⚠ À améliorer — AUC < 0.85")

## 4. Analyse des erreurs

> **Point crucial pour le rapport** : en médical, les faux négatifs (Malignant prédit Benign) sont plus dangereux que les faux positifs.

In [ ]:
find_misclassified(model, test_loader, device, class_names, n_show=8)

## 5. Synthèse des métriques

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

accuracy = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average='macro')
f1_weighted = f1_score(labels, preds, average='weighted')

print("=" * 50)
print("MÉTRIQUES FINALES — SET DE TEST")
print("=" * 50)
print(f"Accuracy   : {accuracy:.4f} ({accuracy:.1%})")
print(f"AUC-ROC    : {roc_auc:.4f}")
print(f"F1 Macro   : {f1_macro:.4f}")
print(f"F1 Weighted: {f1_weighted:.4f}")

# Analyse faux négatifs (critique en médical)
fn = ((labels == 1) & (preds == 0)).sum()  # Malignant prédit Benign
fp = ((labels == 0) & (preds == 1)).sum()  # Benign prédit Malignant
print(f"\nFaux négatifs (Malignant→Benign) : {fn}  ⚠ Critique")
print(f"Faux positifs (Benign→Malignant) : {fp}")
print("=" * 50)